# Duplicate Question JSD Analysis (Single-Label)

This experiment evaluates internal labeling stability on duplicated survey items.

For each question (`qid`), multiple paraphrased variants exist.
We measure how consistently these variants are assigned to the same
wellness dimension.

We compare two systems:

- Human annotators (single-label assignments)
- LLM semantic mapping (Top-1 dimension only)

For each `qid`, dimension assignments across duplicates are converted
into probability distributions over the 8 dimensions.
Jensen–Shannon Divergence (JSD) is then computed to quantify
internal semantic dispersion.

This is not an accuracy comparison.
Instead, it measures internal coherence under duplication.

If LLM exhibits a level of divergence comparable to human annotators,
this supports the structural feasibility of automated survey reduction.

In [1]:
from pathlib import Path
from mhdr.dataloader.io import read_csv, save_csv
INPUT_DIR = Path.cwd() / "input"
OUTPUT_DIR = Path.cwd() / "output"
TEMP_DIR = Path.cwd() / "temp"
HUMAN_DATA = read_csv(INPUT_DIR / "human_dup.csv")


In [3]:
from mhdr.model.semantic_mapper import SemanticMapper
from mhdr.model.LLMio import load_dimension_sets
import pandas as pd
dimension_sets = load_dimension_sets(INPUT_DIR / "dim_definations.csv")

deltas = [0.05]

# 1) init once
mapper = SemanticMapper()
mapper.set_questions_df(HUMAN_DATA, text_col="text_norm")  # encode once

all_results = []

for model_name, dim_defs in dimension_sets.items():
    print(f"\n=== Mapping using {model_name} ===")
    mapper.set_dimensions(dim_defs, dimension_model_name=model_name)

    for delta in deltas:
        mapped = mapper.map_questions_to_dimensions(delta=delta)
        mapped = mapped.copy()
        mapped["delta"] = float(delta)
        all_results.append(mapped)

# 2) merge
all_mappe = pd.concat(all_results, ignore_index=True)
all_mappe = mapper.extract_top1_from_mapped(all_mappe, text_col='text_norm')
save_csv(all_mappe, TEMP_DIR / f"llm_dup.csv")

# 3) sanity checks
print("\nMerged shape:", all_mappe.shape)
print("Unique models:", all_mappe["source"].unique())
print("Unique dimensions:", all_mappe["answer"].nunique())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1683.17it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== Mapping using ChatGPT-5.2 ===

=== Mapping using DeepSeek-V3.2 ===

=== Mapping using Llama-4 ===

=== Mapping using claude-sonnet-4.5 ===

=== Mapping using gemini-3.0-pro ===

Merged shape: (7280, 4)
Unique models: <StringArray>
[      'ChatGPT-5.2',     'DeepSeek-V3.2',           'Llama-4',
 'claude-sonnet-4.5',    'gemini-3.0-pro']
Length: 5, dtype: str
Unique dimensions: 8


In [ ]:
from mhdr.evaluator.internal_evaluator import InternalEvaluator
ev = InternalEvaluator(HUMAN_DATA, all_mappe)

(h_runs, h_sum, _), (l_runs, l_sum, _) = ev.compare_human_vs_llm(
    k_sources=5,   # human sampled to 5, llm sampled to 5 (if llm exactly 5 -> full set)
    times=100,
    seed=42,
    qid_col="qid",
    source_col="source",
    label_col="answer",
)

print(h_sum)
print(l_sum)

   which  k_sources  times      mean    median       p10     p90
0  human          5    100  0.071567  0.065434  0.036722  0.1134
  which  k_sources  times  mean  median  p10  p90
0   llm          5    100   0.0     0.0  0.0  0.0
